# CERT r4.2 insider-threat model

This notebook loads the delivered LightGBM model, inspects the exact features it was trained on, scores genuine training rows, and shows how to reproduce training from the full processed feature table. Each row is one `(user, day)` observation; `label=1` marks an insider-threat day.

In [1]:
from pathlib import Path
import json
import pickle

import pandas as pd
from sklearn.metrics import average_precision_score, roc_auc_score

cwd = Path.cwd()
DELIVERY_DIR = cwd if (cwd / 'insider_threat_supervised_model.pkl').exists() else cwd / 'deliverables'
MODEL_PATH = DELIVERY_DIR / 'insider_threat_supervised_model.pkl'
SAMPLE_PATH = DELIVERY_DIR / 'training_data_sample.csv'
FEATURES_PATH = DELIVERY_DIR / 'model_features.csv'
DELIVERY_DIR

WindowsPath('C:/Users/fadym/Downloads/file-classifier-claude-suspicious-file-access-detection-ak4anr/deliverables')

## Load the trained model and its metadata

In [2]:
with MODEL_PATH.open('rb') as handle:
    bundle = pickle.load(handle)

model = bundle['model']
feature_names = bundle['feature_names']
pd.Series({k: v for k, v in bundle.items() if k not in {'model', 'feature_names'}}, name='value')

artifact_version                                                             1
model_type                                                      LGBMClassifier
model_library                                                         lightgbm
model_library_version                                                    4.6.0
target                                                                   label
entity_key                                                         [user, day]
training_source              CERT Insider Threat Dataset r4.2 processed to ...
training_cutoff_inclusive                                  2010-10-29T00:00:00
training_rows                                                           208167
training_positive_rows                                                     770
random_state                                                                42
held_out_metrics             {'n': 122285, 'positives': 594, 'pr_auc': 0.97...
Name: value, dtype: object

## Exact features used by the supervised model

In [3]:
feature_catalog = pd.read_csv(FEATURES_PATH)
assert feature_catalog['feature'].tolist() == feature_names
feature_catalog

,position,feature,family,description
0,1,logon_count,logon activity,Logon count
1,2,logon_offhours,logon activity,Logon offhours
2,3,logon_first_hour,logon activity,Logon first hour
3,4,logon_last_hour,logon activity,Logon last hour
4,5,device_connects,device activity,Device connects
5,6,device_offhours,device activity,Device offhours
6,7,file_count,file activity,File count
7,8,file_unique,file activity,File unique
8,9,file_offhours,file activity,File offhours
9,10,file_exe,file activity,File exe


## Score the delivered data sample

The sample contains real rows from the training partition and is stratified to include enough positive examples for a practical demonstration. It does not reflect the natural class prevalence.

In [4]:
sample = pd.read_csv(SAMPLE_PATH, parse_dates=['day'])
X_sample = sample[feature_names]
sample['model_score'] = model.predict_proba(X_sample)[:, 1]
sample[['user', 'day', 'label', 'model_score']].sort_values('model_score', ascending=False).head(20)

,user,day,label,model_score
965,RAB0589,2010-09-23,1,1.0
374,BTL0226,2010-10-12,1,1.0
959,VSS0154,2010-09-13,1,1.0
247,PSF0133,2010-09-01,1,1.0
57,PSF0133,2010-08-06,1,1.0
61,PSF0133,2010-08-25,1,1.0
486,AAF0535,2010-08-10,1,1.0
138,HXL0968,2010-09-07,1,1.0
210,HXL0968,2010-09-09,1,1.0
804,RAR0725,2010-07-09,1,1.0


In [5]:
print('Sample PR-AUC:', average_precision_score(sample['label'], sample['model_score']))
print('Sample ROC-AUC:', roc_auc_score(sample['label'], sample['model_score']))

Sample PR-AUC: 1.0
Sample ROC-AUC: 1.0


## Inspect learned feature importance

In [6]:
(pd.Series(model.feature_importances_, index=feature_names, name='split_importance')
 .sort_values(ascending=False).head(20).to_frame())

,split_importance
http_count_dt_z,1483
http_count_z,1196
http_count,599
email_attach_bytes_z,475
email_external_z,474
email_attach_bytes,463
device_connects_z,435
logon_first_hour,421
a,403
file_count_z,348


## Optional: reproduce model training

Set `RUN_TRAINING = True` when the repository's full `artifacts/r4.2/features.parquet` table is available. The split is chronological: the first 60% of calendar days are training data and the remainder is held out. Peer z-score columns are intentionally excluded from the supervised track, matching the production pipeline.

In [7]:
RUN_TRAINING = False

if RUN_TRAINING:
    from insider_threat.evaluate import evaluate, time_split
    from insider_threat.features import feature_matrix
    from insider_threat.models import SupervisedDetector

    repo_root = DELIVERY_DIR.parent
    full_data = pd.read_parquet(repo_root / 'artifacts' / 'r4.2' / 'features.parquet')
    train_mask, cutoff = time_split(full_data, train_frac=0.60)
    X_all, all_numeric_features = feature_matrix(full_data)
    supervised_features = [c for c in all_numeric_features if not c.endswith('_peer_z')]
    X_train = X_all.loc[train_mask, supervised_features]
    y_train = full_data.loc[train_mask, 'label'].to_numpy()
    X_test = X_all.loc[~train_mask, supervised_features]
    y_test = full_data.loc[~train_mask, 'label'].to_numpy()

    detector = SupervisedDetector(random_state=42).fit(X_train, y_train)
    test_scores = detector.score(X_test)
    reproduced_metrics = evaluate(
        test_scores, y_test, full_data.loc[~train_mask, 'user'], top_k=100
    )
    print('Cutoff:', cutoff)
    print(json.dumps(reproduced_metrics, indent=2))